In [ ]:
!pip install polars pyarrow scikit-learn matplotlib seaborn

In [ ]:
import polars as pl

# Carregar o arquivo leve de features prontas
df = pl.read_parquet("features/ml_features_1m_v2.parquet")
print(df.shape)
df.head()

In [ ]:
# Obter o esquema do DataFrame
schema = df.collect_schema()

for col, dtype in df.schema.items():
    print(f"{col}: {dtype}")

In [ ]:
# Obter estatísticas descritivas do DataFrame
df.describe()

In [ ]:
# Verificação da distribuição da variável alvo (target) no dataset.
# Objetivo: Avaliar o balanceamento entre as classes para direcionar a estratégia de validação e escolha de métricas.
# Resultado: Identificado desbalanceamento ~73% (Classe 0) vs ~27% (Classe 1).
# Implicação: Acurácia pura será uma métrica enganosa; usaremos ROC-AUC e F1-Score para avaliar os modelos.
print(df.group_by("target").len())

df.group_by("target").len().with_columns(
    (
        pl.col("len") /
        pl.col("len").sum()
    ).alias("proportion")
)

In [ ]:
# Verificar nulos
df.null_count()

In [ ]:
# Verificar duplicatas
print(
    f"Duplicatas: {df.is_duplicated().sum()}"
)

In [ ]:
# Verificar valores infinitos

numeric_cols = [
    c for c, t in df.schema.items()
    if t in (
        pl.Float32,
        pl.Float64
    )
]

for col in numeric_cols:
    print(
        col,
        df.filter(
            pl.col(col).is_infinite()
        ).height
    )

In [ ]:
# Verificar se o target é binário
df.select(
    pl.col("target").unique().sort()
)

In [ ]:
# Análise estatística da densidade do histórico de dados por mercado (linhas por market_id).
# Objetivo: Identificar a quantidade de observações temporais por ativo para filtrar mercados com histórico insuficiente.
# Resultado: Alta dispersão (média ~1186, mas mediana é 662). Há mercados com apenas 1 minuto de dados (min=1) e mercados altamente líquidos com até 8.221 minutos (max=8221).
# Implicação: Obrigatório aplicar um filtro de corte (ex: manter apenas mercados com len >= 1440 minutos / 1 dia completo) para garantir dados suficientes para o treinamento de modelos sequenciais e exibição no dashboard.
print(df.group_by("market_id").len().describe())
df.group_by("market_id").len().sort("len").head()
print(df.group_by("market_id").len().describe())

df.group_by("market_id").len().sort("len").head()

A análise da distribuição de observações por mercado identificou 4.710 mercados distintos. Embora a maioria possua centenas de observações temporais, alguns mercados apresentam apenas poucas barras de negociação (entre 1 e 4 observações). Esses registros foram mantidos nesta etapa por não representarem erros de qualidade dos dados. A eventual exclusão de mercados com histórico insuficiente será avaliada em etapas posteriores de modelagem.

In [ ]:
# Veificar datas
# Objetivo: Identificar as datas de início e fim do intervalo para planejar a divisão cronológica de treino e teste.
# Resultado: O subconjunto de dados cobre exatamente 6 dias completos, de 06/03/2026 a 11/03/2026.
# Implicação: Ideal para validação inicial (ex: usar os primeiros 4 dias para treino e os 2 últimos para teste).
df.select(
    pl.min("minute_bar").alias("inicio"),
    pl.max("minute_bar").alias("fim")
)

In [ ]:
# Contagem de mercados únicos presentes no subset de dados.
# Objetivo: Identificar a quantidade de ativos/contratos individuais disponíveis para mapear a cardinalidade dos dados.
# Resultado: Encontrados 4.710 mercados distintos ativos no período analisado.
# Implicação: Como cada mercado tem seu próprio comportamento, precisaremos agrupar por 'market_id' ao criar features temporais ou selecionar os mais líquidos para o dashboard.
df.select(
    pl.col("market_id").n_unique()
)

In [ ]:
#A quantidade de linhas ao longo do tempo foi diminuindo. 

(
    df.select(
        pl.col("minute_bar").dt.date().alias("date")
    )
    .group_by("date")
    .len()
    .sort("date")
)